# Quadrant Classification and Mental Health Analysis

Classifies subjects into four quadrants based on the signs of PC1_pace and PC1_status,
then tests whether quadrant membership predicts mental health outcomes.
Includes ABCD discovery analyses and GUSTO replication.

## Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# All paths are relative to the repository root.

# ABCD inputs
FINALSCORE_CSV = 'data/pca/finalscore.csv'                                    # cols: ID, PC1_pace, PC1_status
MH_T1_CSV      = 'data/mental_health/mental_health_ses-00A_CLEAN.csv'         # col:  mh_p_cbcl_sum  (ID index)
MH_T2_CSV      = 'data/mental_health/mental_health_ses-02A_CLEAN.csv'         # cols: mh_p_cbcl_sum, mh_p_cbcl__synd__ext_sum, mh_p_cbcl__synd__int_sum
MH_T3_CSV      = 'data/mental_health/mental_health_ses-04A_CLEAN.csv'
MH_T4_CSV      = 'data/mental_health/mental_health_ses-06A_CLEAN.csv'

# GUSTO replication input
GUSTO_CSV      = 'data/external/final_gusto.csv'                              # cols: ID, PC1_Baseline_Score, PC1_Score, ysr_tot

OUTPUT_DIR     = 'outputs/quadrant_classification'

# Analysis settings
N_BOOT = 5000
N_PERM = 5000
SEED   = 42

## Setup: Imports and Output Directory

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

## Step 1: Load and Merge ABCD Data

In [ ]:
# Load PCA scores
finalscore = pd.read_csv(FINALSCORE_CSV).set_index('ID')

# Load mental health at each timepoint (ID as index)
mh0 = pd.read_csv(MH_T1_CSV).set_index('ID')[['mh_p_cbcl_sum']]
mh2 = pd.read_csv(MH_T2_CSV).set_index('ID')[['mh_p_cbcl_sum', 'mh_p_cbcl__synd__ext_sum', 'mh_p_cbcl__synd__int_sum']]
mh4 = pd.read_csv(MH_T3_CSV).set_index('ID')[['mh_p_cbcl_sum']]
mh6 = pd.read_csv(MH_T4_CSV).set_index('ID')[['mh_p_cbcl_sum']]

# Compute change scores: T2/T3/T4 minus T1
mh2 = mh2.copy(); mh2['change'] = mh2['mh_p_cbcl_sum'] - mh0['mh_p_cbcl_sum']
mh4 = mh4.copy(); mh4['change'] = mh4['mh_p_cbcl_sum'] - mh0['mh_p_cbcl_sum']
mh6 = mh6.copy(); mh6['change'] = mh6['mh_p_cbcl_sum'] - mh0['mh_p_cbcl_sum']

# Merge: inner join on ID
final2 = pd.concat([finalscore, mh2], axis=1, join='inner')
final4 = pd.concat([finalscore, mh4], axis=1, join='inner')
final6 = pd.concat([finalscore, mh6], axis=1, join='inner')

print(f"finalscore  : {finalscore.shape}")
print(f"final2 (T2) : {final2.shape}")
print(f"final4 (T3) : {final4.shape}")
print(f"final6 (T4) : {final6.shape}")

## Step 2: Quadrant Assignment

In [ ]:
def assign_quadrant(x_val, y_val):
    """Assign one of four quadrant labels based on sign of x and y."""
    if   x_val >= 0 and y_val >= 0: return "Positive-Positive"   # Q1
    elif x_val <  0 and y_val >= 0: return "Negative-Positive"   # Q2
    elif x_val >= 0 and y_val <  0: return "Positive-Negative"   # Q3
    else:                            return "Negative-Negative"   # Q4

def add_quadrant_column(df, pace_col='PC1_pace', status_col='PC1_status'):
    """Z-score both PCs within the dataframe, then assign quadrant labels."""
    df = df.copy()
    df['pace_z']   = (df[pace_col]   - df[pace_col].mean())   / df[pace_col].std()
    df['status_z'] = (df[status_col] - df[status_col].mean()) / df[status_col].std()
    df['group'] = df.apply(lambda r: assign_quadrant(r['pace_z'], r['status_z']), axis=1)
    return df

# Apply to the primary T2 merge
final = add_quadrant_column(final2)

print("Quadrant counts (ABCD T2):")
print(final['group'].value_counts().to_string())

## Step 3: One-vs-Rest Group Comparisons (ABCD)

In [ ]:
outcomes   = ['mh_p_cbcl__synd__ext_sum', 'mh_p_cbcl__synd__int_sum', 'mh_p_cbcl_sum']
group_col  = 'group'
groups     = final[group_col].dropna().unique()

rows = []
for var in outcomes:
    df_var = final[[group_col, var]].dropna().copy()
    for g in groups:
        x = df_var.loc[df_var[group_col] == g, var]
        y = df_var.loc[df_var[group_col] != g, var]
        t_stat, p_raw = ttest_ind(x, y, equal_var=False)   # Welch's t-test
        rows.append({
            'outcome':  var,
            'quadrant': g,
            'n_group':  len(x),
            'n_rest':   len(y),
            'mean_group': x.mean(),
            'mean_rest':  y.mean(),
            't_stat':   t_stat,
            'p_raw':    p_raw,
        })

df_ovr = pd.DataFrame(rows)

# FDR correction (Benjamini-Hochberg) per outcome
df_ovr['p_fdr'] = np.nan
for var in outcomes:
    mask = df_ovr['outcome'] == var
    _, pvals_corr, _, _ = multipletests(df_ovr.loc[mask, 'p_raw'], method='fdr_bh')
    df_ovr.loc[mask, 'p_fdr'] = pvals_corr

out_path = os.path.join(OUTPUT_DIR, 'abcd_group_comparisons.csv')
df_ovr.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(df_ovr.to_string(index=False))

## Step 4: Pairwise Mann-Whitney Between Quadrants (ABCD)

In [ ]:
rows_pw = []
for var in outcomes:
    df_var = final[[group_col, var]].dropna().copy()
    group_list = sorted(df_var[group_col].unique())
    for g1, g2 in combinations(group_list, 2):
        x1 = df_var.loc[df_var[group_col] == g1, var]
        x2 = df_var.loc[df_var[group_col] == g2, var]
        stat, p_raw = mannwhitneyu(x1, x2, alternative='two-sided')
        rows_pw.append({
            'outcome':   var,
            'group_1':   g1,
            'group_2':   g2,
            'n_1':       len(x1),
            'n_2':       len(x2),
            'mwu_stat':  stat,
            'p_raw':     p_raw,
        })

df_pw = pd.DataFrame(rows_pw)

# FDR correction per outcome
df_pw['p_fdr'] = np.nan
for var in outcomes:
    mask = df_pw['outcome'] == var
    _, pvals_corr, _, _ = multipletests(df_pw.loc[mask, 'p_raw'], method='fdr_bh')
    df_pw.loc[mask, 'p_fdr'] = pvals_corr

out_path = os.path.join(OUTPUT_DIR, 'abcd_pairwise_mannwhitney.csv')
df_pw.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(df_pw.to_string(index=False))

## Step 5: Bootstrap CI for Group Means (ABCD)

In [ ]:
def bootstrap_mean_ci(x, n_boot=5000, ci=95, seed=42):
    """Return (lower, upper) bootstrap percentile CI for the mean."""
    rng  = np.random.default_rng(seed)
    boot = [np.mean(rng.choice(x, size=len(x), replace=True)) for _ in range(n_boot)]
    return np.percentile(boot, [(100 - ci) / 2, 100 - (100 - ci) / 2])

rows_ci = []
for var in outcomes:
    df_var = final[[group_col, var]].dropna().copy()
    for g in sorted(df_var[group_col].unique()):
        x  = df_var.loc[df_var[group_col] == g, var].values
        lo, hi = bootstrap_mean_ci(x, n_boot=N_BOOT, ci=95, seed=SEED)
        rows_ci.append({
            'outcome':  var,
            'quadrant': g,
            'n':        len(x),
            'mean':     np.mean(x),
            'ci_lower': lo,
            'ci_upper': hi,
        })

df_ci = pd.DataFrame(rows_ci)

out_path = os.path.join(OUTPUT_DIR, 'abcd_bootstrap_ci.csv')
df_ci.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(df_ci.to_string(index=False))

## Step 6: GUSTO Replication

In [ ]:
# Load GUSTO and align column names to match the ABCD convention
gusto = pd.read_csv(GUSTO_CSV).set_index('ID')
gusto = gusto.rename(columns={
    'PC1_Baseline_Score': 'PC1_status',
    'PC1_Score':          'PC1_pace',
})

# Apply the same quadrant assignment
gusto = add_quadrant_column(gusto, pace_col='PC1_pace', status_col='PC1_status')

print("Quadrant counts (GUSTO):")
print(gusto['group'].value_counts().to_string())

In [ ]:
# One-vs-rest Mann-Whitney for GUSTO outcome (ysr_tot)
gusto_outcome = 'ysr_tot'
df_g = gusto[[group_col, gusto_outcome]].dropna().copy()
g_groups = df_g[group_col].unique()

rows_g = []
for g in g_groups:
    x = df_g.loc[df_g[group_col] == g, gusto_outcome]
    y = df_g.loc[df_g[group_col] != g, gusto_outcome]
    stat, p_raw = mannwhitneyu(x, y, alternative='two-sided')
    rows_g.append({
        'outcome':    gusto_outcome,
        'quadrant':   g,
        'n_group':    len(x),
        'n_rest':     len(y),
        'mean_group': x.mean(),
        'mean_rest':  y.mean(),
        'mwu_stat':   stat,
        'p_raw':      p_raw,
    })

df_gusto_ovr = pd.DataFrame(rows_g)

# FDR correction
_, pvals_corr, _, _ = multipletests(df_gusto_ovr['p_raw'], method='fdr_bh')
df_gusto_ovr['p_fdr'] = pvals_corr

out_path = os.path.join(OUTPUT_DIR, 'gusto_group_comparisons.csv')
df_gusto_ovr.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(df_gusto_ovr.to_string(index=False))